# Create and Consolidate ESM-DB Metadata

**Author:** Spina Cianetti  
**License:** [GPL-3.0](https://www.gnu.org/licenses/gpl-3.0.html)  
This code is released under the GNU General Public License v3.0.

---

This notebook reads the per-event flatfile CSVs downloaded from ESM-DB (one `_SA.csv` and one
`_SD.csv` per event), merges the SA and SD spectral metadata into a single consolidated table,
applies a standardised column naming convention, and splits the output into three files:

| Output file | Contents |
|-------------|----------|
| `metadata_all.csv` | All records; multi-segment fault rows are aggregated into lists |
| `metadata_good.csv` | Processed (good quality) records only |
| `metadata_bad.csv` | Bad-quality records only |

**Column naming convention** — four domain prefixes are used throughout:

| Prefix | Domain |
|--------|--------|
| `source_*` | Earthquake source parameters (origin, magnitude, fault geometry) |
| `station_*` | Recording station metadata (location, site conditions) |
| `path_*` | Source-to-site distance and azimuth metrics |
| `trace_*` | Waveform parameters, IMs, filter corners, spectral ordinates |

**Sections:**
1. Setup and configuration
2. Read and merge SA / SD flatfiles
3. Column renaming
4. Derive `trace_name` identifier
5. Aggregate multi-segment fault rows
6. Save consolidated output
7. Split by processing quality
8. Diagnostics

## 1. Setup and configuration

In [ ]:
import os
import glob
import re

import numpy as np
import pandas as pd

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_DIR = '/home/jovyan/shared/users/spina/ESM25/output/'

# Output files
OUT_ALL  = 'metadata_all.csv'
OUT_GOOD = 'metadata_good.csv'
OUT_BAD  = 'metadata_bad.csv'

# Dtype overrides: preserve leading zeros in SEED code fields
DTYPE_OVERRIDES = {
    'station_code':   str,
    'network_code':   str,
    'channel_code':   str,
    'location_code':  str,
    'u_channel_code': str,
    'v_channel_code': str,
    'w_channel_code': str,
}

# Columns excluded from the SA–SD merge key.
# These fields are present in both files but carry the same values;
# excluding them avoids '_SA' / '_SD' suffixes on non-spectral columns.
EXCLUDE_FROM_KEYS = [
    'epi_dist', 'epi_az', 'hyp_dist', 'jb_dist', 'rup_dist',
    'rx_dist', 'ry0_dist',
    'st_latitude', 'st_longitude', 'st_elevation',
    'u_azimuth_deg', 'v_azimuth_deg', 'w_azimuth_deg', 'w_inclination_deg',
]

print('Configuration loaded.')

## 2. Read and merge SA / SD flatfiles

Each per-event CSV is read and concatenated into two DataFrames (`df_SA`, `df_SD`).
Spectral ordinate columns matching the pattern `<component>_t<n>_<period>` are renamed
on the fly to include physical unit suffixes:

- `_cps2` for SA (pseudo-spectral acceleration, cm s⁻²)
- `_c` for SD (spectral displacement, cm)

The two DataFrames are then joined with an inner merge on all shared non-spectral
metadata columns. `validate='one_to_one'` enforces that the keys are unique on
both sides, raising an error if duplicates are found.

In [ ]:
def rename_spectral_column(col: str, spectra_type: str) -> str:
    """Add physical-unit suffix to spectral ordinate column names.

    Columns matching `<component>_t<n>_<period>` are renamed to
    `trace_<component>_t<n>_<period>_<unit>`, where *unit* is
    'cps2' for SA (cm s⁻²) and 'c' for SD (cm).
    All other column names are returned unchanged.

    Parameters
    ----------
    col          : Original column name.
    spectra_type : 'SA' or 'SD'.
    """
    m = re.match(r'^(u|v|w|rotd50|rotd100|rotd00)_t(\d+)_([0-9]{3})$', col)
    if m:
        comp, tnum, period = m.groups()
        unit = 'cps2' if spectra_type == 'SA' else 'c'
        return f'trace_{comp}_t{tnum}_{period}_{unit}'
    return col


def read_flatfiles(pattern: str, spectra_type: str) -> pd.DataFrame:
    """Read and concatenate all per-event flatfile CSVs matching *pattern*.

    Parameters
    ----------
    pattern      : Glob pattern relative to INPUT_DIR (e.g. '*_SA.csv').
    spectra_type : 'SA' or 'SD' — determines spectral column unit suffix.

    Returns
    -------
    Concatenated DataFrame, or an empty DataFrame if no files are found.
    """
    files = sorted(glob.glob(os.path.join(INPUT_DIR, pattern)))
    if not files:
        print(f'[WARNING] No files found for pattern: {pattern}')
        return pd.DataFrame()

    frames = []
    for path in files:
        try:
            df = pd.read_csv(path, sep=';', index_col=False, dtype=DTYPE_OVERRIDES)
            if not df.empty:
                df.columns = [
                    rename_spectral_column(c, spectra_type) for c in df.columns
                ]
                frames.append(df)
        except Exception as e:
            print(f'[ERROR] {path}: {e}')

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


# ── Read ──────────────────────────────────────────────────────────────────────
df_SA = read_flatfiles('*_SA.csv', 'SA')
df_SD = read_flatfiles('*_SD.csv', 'SD')

print(f'SA flatfiles: {df_SA.shape[0]:>8,} rows  {df_SA.shape[1]:>4} columns')
print(f'SD flatfiles: {df_SD.shape[0]:>8,} rows  {df_SD.shape[1]:>4} columns')

In [ ]:
# ── Prepare for merge ─────────────────────────────────────────────────────────
# Drop the optional tracking column added during download (present only in some runs)
df_SA = df_SA.drop(columns=['source_file'], errors='ignore')
df_SD = df_SD.drop(columns=['source_file'], errors='ignore')

# Replace NaN with empty string so that key comparisons are deterministic
# (NaN != NaN in standard Python equality, which would break the merge)
df_SA = df_SA.replace({np.nan: ''})
df_SD = df_SD.replace({np.nan: ''})

# Merge keys: all non-spectral columns not in the exclusion list
merge_keys = [
    col for col in df_SA.columns
    if not col.startswith('trace') and col not in EXCLUDE_FROM_KEYS
]
print(f'Merge keys ({len(merge_keys)}): {merge_keys}')

# ── Inner join ────────────────────────────────────────────────────────────────
df_merged = pd.merge(
    df_SA,
    df_SD,
    on=merge_keys,
    how='inner',
    suffixes=('_SA', '_SD'),
    validate='one_to_one',   # raises MergeError if keys are not unique
)
print(f'Merged shape: {df_merged.shape[0]:,} rows  {df_merged.shape[1]} columns')

In [ ]:
# ── Remove suffixed duplicate columns ────────────────────────────────────────
# Columns that were identical in SA and SD receive '_SA' / '_SD' suffixes.
# Keep the '_SA' copies (strip the suffix) and drop all '_SD' duplicates.
df_merged.rename(
    columns=lambda x: x[:-3] if x.endswith('_SA') else x,
    inplace=True,
)
cols_to_drop = [c for c in df_merged.columns if c.endswith('_SD')]
df_merged.drop(columns=cols_to_drop, inplace=True)

print(f'Shape after deduplication: {df_merged.shape[0]:,} rows  {df_merged.shape[1]} columns')

## 3. Column renaming

All columns are renamed to follow the four-prefix convention described in the
header. A post-rename sanity check verifies that every column now starts with
one of the expected prefixes and prints any unmapped remainder.

In [ ]:
# ── Rename map ───────────────────────────────────────────────────────────────
# Organised by domain for readability. Unit suffixes are included where the
# original name was ambiguous (e.g. _km, _deg, _mps, _cps2, _c, _s).
RENAME_MAP = {
    # ── Source ────────────────────────────────────────────────────────────────
    'esm_event_id':                          'source_esm_id',
    'event_time':                            'source_esm_origin_time',
    'isc_event_id':                          'source_isc_id',
    'usgs_event_id':                         'source_usgs_id',
    'ingv_event_id':                         'source_ingv_id',
    'emsc_event_id':                         'source_emsc_id',
    'ev_nation_code':                        'source_nation_code',
    'ev_latitude':                           'source_latitude_deg',
    'ev_longitude':                          'source_longitude_deg',
    'ev_depth_km':                           'source_depth_km',
    'ev_hyp_ref':                            'source_hyp_ref',
    'fm_type_code':                          'source_fm_type_code',
    'fm_ref':                                'source_fm_ref',
    'ml':                                    'source_magnitude_ml',
    'ml_ref':                                'source_magnitude_ml_ref',
    'mw':                                    'source_magnitude_mw',
    'mw_ref':                                'source_magnitude_mw_ref',
    'ms':                                    'source_magnitude_ms',
    'ms_ref':                                'source_magnitude_ms_ref',
    'mb':                                    'source_magnitude_mb',
    'mb_ref':                                'source_magnitude_mb_ref',
    'md':                                    'source_magnitude_md',
    'md_ref':                                'source_magnitude_md_ref',
    'm':                                     'source_magnitude_m',
    'm_ref':                                 'source_magnitude_m_ref',
    'emec_mw':                               'source_magnitude_emec_mw',
    'emec_mw_type':                          'source_magnitude_emec_mw_type',
    'emec_mw_ref':                           'source_magnitude_emec_mw_ref',
    'event_source_name':                     'source_fault_name',
    'source_segment_id':                     'source_segment_id',
    'es_strike':                             'source_segment_strike_deg',
    'es_dip':                                'source_segment_dip_deg',
    'es_rake':                               'source_segment_rake_deg',
    'es_strike_dip_rake_ref':                'source_segment_strike_dip_rake_ref',
    'es_length':                             'source_segment_length_km',
    'es_width':                              'source_segment_width_km',
    'es_geometry_ref':                       'source_segment_geometry_ref',
    'z_top':                                 'source_segment_z_top_km',
    'es_z_top_ref':                          'source_segment_z_top_ref',
    # ── Station ───────────────────────────────────────────────────────────────
    'network_code':                          'station_network_code',
    'station_code':                          'station_code',
    'location_code':                         'station_location_code',
    'channel_code':                          'station_channel_code',
    'sensor_depth_m':                        'station_sensor_depth_m',
    'proximity':                             'station_proximity',
    'housing':                               'station_housing',
    'installation':                          'station_installation',
    'st_nation_code':                        'station_nation_code',
    'st_latitude':                           'station_latitude_deg',
    'st_longitude':                          'station_longitude_deg',
    'st_elevation':                          'station_elevation_m',
    'reference_site':                        'station_reference_site',
    'preferred_estimation_method_vs30_ec8':  'station_preferred_estimation_method_vs30_ec8',
    'preferred_ec8_code':                    'station_preferred_ec8_code',
    'preferred_vs30_m_s':                    'station_preferred_vs30_mps',
    'vs30_m_s':                              'station_vs30_mps',
    'ec8_code':                              'station_ec8_code',
    'vs30_meas_type':                        'station_vs30_meas_type',
    'vs30_ref_programme':                    'station_vs30_ref_programme',
    'vs30_ref_auth':                         'station_vs30_ref_auth',
    'ec8_code_from_geology':                 'station_ec8_code_from_geology',
    'ec8_ref_programme':                     'station_ec8_ref_programme',
    'ec8_ref_auth':                          'station_ec8_ref_auth',
    'vs30_m_s_wa':                           'station_vs30_mps_wa',
    'ec8_code_from_topography':              'station_ec8_code_from_topography',
    'slope_deg':                             'station_slope_deg',
    'vs30_wa_ref':                           'station_vs30_wa_ref',
    'instrument_type_code':                  'station_instrument_type_code',
    # ── Path ──────────────────────────────────────────────────────────────────
    'epi_dist':                              'path_epi_dist_km',
    'epi_az':                                'path_epi_az_deg',      # azimuth, degrees
    'hyp_dist':                              'path_hyp_dist_km',
    'jb_dist':                               'path_jb_dist_km',
    'rup_dist':                              'path_rup_dist_km',
    'rx_dist':                               'path_rx_dist_km',
    'ry0_dist':                              'path_ry0_dist_km',
    # ── Trace ─────────────────────────────────────────────────────────────────
    'u_channel_code':                        'trace_u_channel_code',
    'u_inclination_deg':                     'trace_u_inclination_deg',
    'u_azimuth_deg':                         'trace_u_azimuth_deg',
    'v_channel_code':                        'trace_v_channel_code',
    'v_inclination_deg':                     'trace_v_inclination_deg',
    'v_azimuth_deg':                         'trace_v_azimuth_deg',
    'w_channel_code':                        'trace_w_channel_code',
    'w_inclination_deg':                     'trace_w_inclination_deg',  # fixed leading-space typo
    'w_azimuth_deg':                         'trace_w_azimuth_deg',
    'u_un_pga':                              'trace_u_un_pga_cps2',
    'v_un_pga':                              'trace_v_un_pga_cps2',
    'w_un_pga':                              'trace_w_un_pga_cps2',
    'processing_status':                     'trace_processing_status',
    'processing_type':                       'trace_processing_type',
    'u_hp':                                  'trace_u_hp_hz',
    'v_hp':                                  'trace_v_hp_hz',
    'w_hp':                                  'trace_w_hp_hz',            # fixed leading-space typo
    'u_lp':                                  'trace_u_lp_hz',
    'v_lp':                                  'trace_v_lp_hz',
    'w_lp':                                  'trace_w_lp_hz',
    'u_pga':                                 'trace_u_pga_cps2',
    'v_pga':                                 'trace_v_pga_cps2',
    'w_pga':                                 'trace_w_pga_cps2',
    'rotd50_pga':                            'trace_rotd50_pga_cps2',
    'rotd100_pga':                           'trace_rotd100_pga_cps2',
    'rotd00_pga':                            'trace_rotd00_pga_cps2',
    'u_pgv':                                 'trace_u_pgv_cps',
    'v_pgv':                                 'trace_v_pgv_cps',
    'w_pgv':                                 'trace_w_pgv_cps',
    'rotd50_pgv':                            'trace_rotd50_pgv_cps',
    'rotd100_pgv':                           'trace_rotd100_pgv_cps',
    'rotd00_pgv':                            'trace_rotd00_pgv_cps',
    'u_pgd':                                 'trace_u_pgd_c',
    'v_pgd':                                 'trace_v_pgd_c',
    'w_pgd':                                 'trace_w_pgd_c',
    'rotd50_pgd':                            'trace_rotd50_pgd_c',
    'rotd100_pgd':                           'trace_rotd100_pgd_c',
    'rotd00_pgd':                            'trace_rotd00_pgd_c',
    'u_t90':                                 'trace_u_t90_s',
    'v_t90':                                 'trace_v_t90_s',
    'w_t90':                                 'trace_w_t90_s',
    'rotd50_t90':                            'trace_rotd50_t90_s',
    'rotd100_t90':                           'trace_rotd100_t90_s',
    'rot_d00_t90':                           'trace_rotd00_t90_s',
    'u_housner':                             'trace_u_housner_c',
    'v_housner':                             'trace_v_housner_c',
    'w_housner':                             'trace_w_housner_c',
    'rotd50_housner':                        'trace_rotd50_housner_c',
    'rotd100_housner':                       'trace_rotd100_housner_c',
    'rotd00_housner':                        'trace_rotd00_housner_c',
    'u_cav':                                 'trace_u_cav_cps',
    'v_cav':                                 'trace_v_cav_cps',
    'w_cav':                                 'trace_w_cav_cps',
    'rotd50_cav':                            'trace_rotd50_cav_cps',
    'rotd100_cav':                           'trace_rotd100_cav_cps',
    'rotd00_cav':                            'trace_rotd00_cav_cps',
    'u_ai':                                  'trace_u_ai_cps',
    'v_ai':                                  'trace_v_ai_cps',
    'w_ai':                                  'trace_w_ai_cps',
    'rotd50_ai':                             'trace_rotd50_ai_cps',   # fixed typo: was 'rtrace_otd50_ai_cps'
    'rotd100_ai':                            'trace_rotd100_ai_cps',
    'rotd00_ai':                             'trace_rotd00_ai_cps',
    'late_triggered_event_01':               'trace_late_triggered_event_01',
    # ── Uncategorised ─────────────────────────────────────────────────────────
    'web_availability_code':                 'web_availability_code',
    'original_data_mediator':                'original_data_mediator',
}

df_finale = df_merged.rename(columns=RENAME_MAP)

# ── Sanity check: flag any column not yet assigned a domain prefix ─────────────
EXPECTED_PREFIXES = ('source_', 'station_', 'path_', 'trace_',
                     'web_', 'original_')
unmapped = [
    c for c in df_finale.columns
    if not any(c.startswith(p) for p in EXPECTED_PREFIXES)
]
if unmapped:
    print(f'[WARNING] {len(unmapped)} column(s) without a recognised prefix:')
    for c in unmapped:
        print(f'  {c}')
else:
    print(f'✅ All {len(df_finale.columns)} columns successfully remapped.')
print(f'DataFrame shape: {df_finale.shape[0]:,} rows  {df_finale.shape[1]} columns')

## 4. Derive `trace_name` identifier

A unique string identifier for each record is built by concatenating the event ID,
network code, station code, location code, and channel code, following the SEED
naming pattern:

```
<source_esm_id>.<network>.<station>.<location>.<channel>
```

This identifier mirrors the waveform tag used in the ASDF files and enables
cross-referencing between the metadata table and the waveform data.

In [ ]:
df_finale['trace_name'] = (
    df_finale['source_esm_id'].astype(str)        + '.' +
    df_finale['station_network_code'].astype(str)  + '.' +
    df_finale['station_code'].astype(str)          + '.' +
    df_finale['station_location_code'].astype(str) + '.' +
    df_finale['station_channel_code'].astype(str)
)

print(f'Unique trace_name values : {df_finale["trace_name"].nunique():,}')
print(f'Example values:')
print(df_finale['trace_name'].head(5).to_string(index=False))

## 5. Aggregate multi-segment fault rows

Some events are associated with a fault source model that comprises multiple
segments (e.g. a multi-segment rupture). The ESM flatfile represents each segment
as a separate row, which means one (event, station, channel) combination may appear
in several rows. This section collapses those rows into one, storing the per-segment
fields as Python lists.

**Steps:**
1. Separate rows that have a fault name from those that do not.
2. Verify that non-segment columns are identical across all rows that share the
   same (event × station × channel × fault) key — an inconsistency would indicate
   a data-integrity problem.
3. Aggregate segment parameters into lists, sorted by `source_segment_id`.
4. Re-join the aggregated fault rows with the non-fault rows.

In [ ]:
# ── Column groups ─────────────────────────────────────────────────────────────
# Keys uniquely identifying a recording for a given fault source
KEY_COLS = [
    'source_esm_id',
    'station_network_code',
    'station_code',
    'station_location_code',
    'station_channel_code',
    'source_fault_name',
]

# Columns that describe a single fault segment (may differ across rows)
SEGMENT_COLS = [
    'source_segment_id',
    'source_segment_strike_deg',
    'source_segment_dip_deg',
    'source_segment_rake_deg',
    'source_segment_length_km',
    'source_segment_width_km',
    'source_segment_z_top_km',
]

# ── Split rows ────────────────────────────────────────────────────────────────
mask_fault = (
    df_finale['source_fault_name'].notna() &
    (df_finale['source_fault_name'].str.strip() != '')
)
df_no_fault = df_finale.loc[~mask_fault].copy()
df_fault    = df_finale.loc[mask_fault].copy()

print(f'Rows without fault info : {len(df_no_fault):,}')
print(f'Rows with    fault info : {len(df_fault):,}')

In [ ]:
# ── Consistency check ────────────────────────────────────────────────────────
# For multi-segment records, non-segment columns must be identical across all rows
# that share the same key. Any variation is flagged as a potential data issue.

# Normalise key strings and convert segment ID to numeric
for col in KEY_COLS:
    df_fault[col] = df_fault[col].astype(str).str.strip()
df_fault['source_segment_id'] = pd.to_numeric(
    df_fault['source_segment_id'], errors='coerce'
)

non_segment_cols = [c for c in df_fault.columns if c not in SEGMENT_COLS + KEY_COLS]
grouped_seg_counts = df_fault.groupby(KEY_COLS)['source_segment_id'].nunique()
multi_segment_keys = grouped_seg_counts[grouped_seg_counts > 1]

inconsistencies = []
for key_vals, _ in multi_segment_keys.items():
    # Select the group using a boolean mask (avoids MultiIndex ambiguity)
    mask = True
    for k, v in zip(KEY_COLS, key_vals):
        mask = mask & (df_fault[k] == v)
    g = df_fault.loc[mask]
    varying = [c for c in non_segment_cols if g[c].nunique(dropna=False) > 1]
    if varying:
        inconsistencies.append({'key': key_vals, 'varying_cols': varying})

print(f'Records with multiple segments : {len(multi_segment_keys):,}')
if inconsistencies:
    print(f'[WARNING] {len(inconsistencies)} key(s) with inconsistent non-segment values:')
    for item in inconsistencies:
        print(f"  Key   : {item['key']}")
        print(f"  Varying: {item['varying_cols']}")
else:
    print('✅ All non-segment columns are consistent across segments.')

In [ ]:
# ── Aggregate segment rows ────────────────────────────────────────────────────
# For each unique (key) group, segment parameters are collected into lists
# (sorted by segment ID) and a single representative value is kept for all
# other columns (consistency verified in the cell above).

non_seg_cols = [c for c in df_fault.columns if c not in SEGMENT_COLS + KEY_COLS]
rows = []

for key_vals, group in df_fault.groupby(KEY_COLS):
    group = group.sort_values('source_segment_id')

    row = dict(zip(KEY_COLS, key_vals))
    row['source_segment_num'] = int(group['source_segment_id'].nunique())

    # Segment parameters → lists (one entry per segment)
    for col in SEGMENT_COLS:
        row[col] = group[col].tolist()

    # All other columns → first row's value
    for col in non_seg_cols:
        row[col] = group[col].iloc[0]

    rows.append(row)

df_fault_agg = pd.DataFrame(rows)
print(f'Fault rows before aggregation : {len(df_fault):,}')
print(f'Fault rows after  aggregation : {len(df_fault_agg):,}')

## 6. Save consolidated output

Re-join the aggregated fault rows with the non-fault rows, sort by event ID and
station code, and write the full table to `metadata_all.csv`.

In [ ]:
df_updated = pd.concat(
    [df_no_fault, df_fault_agg], ignore_index=True
)
df_updated = df_updated.sort_values(
    ['source_esm_id', 'station_code', 'source_fault_name'],
    na_position='last',
).reset_index(drop=True)

df_updated.to_csv(OUT_ALL, index=False)

print(f'✅  {OUT_ALL} saved')
print(f'    Rows            : {df_updated.shape[0]:,}')
print(f'    Columns         : {df_updated.shape[1]}')
print(f'    Unique events   : {df_updated["source_esm_id"].nunique():,}')
print(f'    Unique stations : {df_updated["station_code"].nunique():,}')

## 7. Split by processing quality

The full table is filtered on `trace_processing_status` into two subsets:
- **`processed`** → `metadata_good.csv`
- **`bad quality record`** → `metadata_bad.csv`

Columns that are entirely empty within a subset are removed before saving, as they
carry no information for that partition.

In [ ]:
def drop_empty_columns(df: pd.DataFrame, label: str = '') -> pd.DataFrame:
    """Drop columns that are wholly empty (NaN or whitespace-only strings).

    Parameters
    ----------
    df    : Input DataFrame.
    label : Optional label for the printed message (e.g. 'good', 'bad').
    """
    tmp = df.astype(object)
    empty = [
        col for col in tmp.columns
        if tmp[col].replace(r'^\s*$', np.nan, regex=True).isna().all()
    ]
    if empty:
        tag = f' [{label}]' if label else ''
        print(f'  Dropping {len(empty)} wholly-empty column(s){tag}: {empty}')
    return df.drop(columns=empty)


# ── Bad quality ───────────────────────────────────────────────────────────────
df_bad = df_updated[
    df_updated['trace_processing_status'] == 'bad quality record'
].copy()
df_bad = drop_empty_columns(df_bad, label='bad')
df_bad.to_csv(OUT_BAD, index=False)
print(f'✅  {OUT_BAD} saved  ({len(df_bad):,} rows, {df_bad.shape[1]} columns)')

# ── Good quality ──────────────────────────────────────────────────────────────
df_good = df_updated[
    df_updated['trace_processing_status'] == 'processed'
].copy()
df_good = drop_empty_columns(df_good, label='good')
df_good.to_csv(OUT_GOOD, index=False)
print(f'✅  {OUT_GOOD} saved  ({len(df_good):,} rows, {df_good.shape[1]} columns)')

## 8. Diagnostics

Quick inspection of quality-flag columns and channel codes to verify the output
is consistent with expectations.

In [ ]:
# ── Processing status and type ───────────────────────────────────────────────
print('── trace_processing_status (all records) ──────────────────────────────')
print(df_updated['trace_processing_status'].value_counts(dropna=False).to_string())

print('\n── trace_processing_type (processed records only) ─────────────────────')
print(df_good['trace_processing_type'].value_counts(dropna=False).to_string())

print('\n── web_availability_code (all records) ────────────────────────────────')
print(df_updated['web_availability_code'].value_counts(dropna=False).to_string())

In [ ]:
# ── Component channel codes ───────────────────────────────────────────────────
# Each of the three rotated components (u, v, w) should map to a well-defined
# set of SEED channel codes. Unexpected values may indicate a data anomaly.
for comp in ('u', 'v', 'w'):
    col = f'trace_{comp}_channel_code'
    if col in df_finale.columns:
        vals = sorted(df_finale[col].dropna().unique())
        print(f'{col}: {vals}')

In [ ]:
# ── Record-level summary ──────────────────────────────────────────────────────
print(f'Total records in metadata_all  : {len(df_updated):>10,}')
print(f'  of which processed (good)    : {len(df_good):>10,}')
print(f'  of which bad quality         : {len(df_bad):>10,}')
print(f'  unaccounted                  : '
      f'{len(df_updated) - len(df_good) - len(df_bad):>10,}')
print(f'Unique source events           : {df_updated["source_esm_id"].nunique():>10,}')
print(f'Unique stations                : {df_updated["station_code"].nunique():>10,}')